In [1]:
import os
import sys
import time
import yaml
import pandas as pd
import numpy as np
import re

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import data_tools as dt
import writing_tools as wt
from utils import parse_casenum

from matplotlib import pyplot as plt
from sklearn.linear_model import LogisticRegression
from IPython.core.display import HTML
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import statsmodels.api as sm
from stargazer.stargazer import Stargazer

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11

os.environ['LOKY_MAX_CPU_COUNT'] = '1' # because of windows core count warning

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)
with open('../../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']
EMBEDDING_DIMENSION = config['EMBEDDING_DIMENSION']

rng = np.random.default_rng(12898)

N_CLUSTERS = 3
N_COMPONENTS = 10


In [2]:
meetings_df = pd.read_csv(os.path.join(DATA_PATH, "intermediate_data/cpc/meetings-manifest.csv"))

agenda_df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "agenda_items.parquet"))

docs_df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "supplemental_docs.parquet"))

df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "analysis_data_w_embeddings.parquet"))

In [3]:
# Basic info

NUMBER_OF_MEETINGS = meetings_df['date'].nunique()
FIRST_MEETING_DATE = pd.to_datetime(meetings_df['date']).min()
LAST_MEETING_DATE = pd.to_datetime(meetings_df['date']).max()
NUMBER_OF_YEARS = LAST_MEETING_DATE.year - FIRST_MEETING_DATE.year + 1
NUMBER_OF_CASES = len(df)

RESULTS = {
    "NumberOfMeetings": f"{NUMBER_OF_MEETINGS}",
    "FirstMeetingDate": FIRST_MEETING_DATE.strftime('%Y-%m-%d'),
    "LastMeetingDate": LAST_MEETING_DATE.strftime('%Y-%m-%d'),
    "NumberOfYears": f"{NUMBER_OF_YEARS}",
    "NumberOfCases": f"{NUMBER_OF_CASES:,.0f}"
}

_ = wt.update_results(RESULTS)
RESULTS


{'NumberOfMeetings': '156',
 'FirstMeetingDate': '2018-05-10',
 'LastMeetingDate': '2024-12-19',
 'NumberOfYears': '7',
 'NumberOfCases': '727'}

In [4]:
# Number of agenda items (total, including non-cases)

AGENDA_ITEMS = len(agenda_df)

RESULTS = {
    "NumberOfAgendaItems": f"{AGENDA_ITEMS:,.0f}"
}

_ = wt.update_results(RESULTS)
RESULTS

{'NumberOfAgendaItems': '1,542'}

In [5]:
# Number of (usable) supplemental documents

SUPPLEMENTAL_DOCS = len(docs_df)

RESULTS = {
    "NumberOfSupplementalDocs": f"{SUPPLEMENTAL_DOCS:,.0f}"
}

_ = wt.update_results(RESULTS)
RESULTS


{'NumberOfSupplementalDocs': '5,390'}

In [6]:
# Page counts

PAGE_COUNT = meetings_df["agenda_pages"].sum() + meetings_df["minutes_pages"].sum() + meetings_df["supdocs_pages"].sum()
RESULTS = {
    "PageCount": f"{PAGE_COUNT:,.0f}"
}
_ = wt.update_results(RESULTS)
RESULTS
    

{'PageCount': '23,712'}

In [9]:
df['project_result'].fillna('na').value_counts()

project_result
APPROVED                                           299
APPROVED WITH MINOR CONDITIONS OR MODIFICATIONS    279
DELAYED                                            110
APPROVED WITH MAJOR CONDITIONS OR MODIFICATIONS     27
DENIED                                               8
APPLICATION WITHDRAWN                                4
Name: count, dtype: int64

In [7]:
# Motion results vs. unanimity

df['unanimity'] = ''
df.loc[ df['num_noes']==0, 'unanimity'] = 'Unanimous'
df.loc[ df['num_noes']==1, 'unanimity'] = '1 Nay'
df.loc[ df['num_noes']>1, 'unanimity'] = '>1 Nays'

header = r"""
\begin{table}[H]
\caption{Summary of Motion Outcomes and Vote Results}
\vspace{0.2cm}
\label{tab_result_unanimity}
\begin{adjustbox}{max width=\textwidth}
\begin{threeparttable}
\begin{tabular}{lrrrrr} \toprule
 & \multicolumn{3}{c}{Unanimity} & & \\
 Project Implication & Unanimous & 1 Nay & >1 Nays & Total &  \\ \midrule
 """
footer = r"""
\bottomrule
\end{tabular}
\begin{tablenotes}
\item {\textit{Notes: } This table shows the number of cases decided on by the City Planning Commission, organized by the implication of the motion for the project proposal and the unanimity of the vote.}
\end{tablenotes}
\end{threeparttable}
\end{adjustbox}
\end{table}
"""
tbl = ""

# Approved
tbl += " ~ ~ Approved"
mask = df['project_result'] == 'APPROVED'
for u in ["Unanimous", "1 Nay", ">1 Nays"]:
    tbl += f" & {(df.loc[mask, 'unanimity']==u).sum():,.0f}"
tbl += f" & {mask.sum():,.0f}"
tbl += f" & ({mask.sum()/len(df):.1%}) \\\\ [1ex]\n"

# Approved with minor conditions
tbl += " ~ ~ Approved with minor conditions"
mask = df['project_result'] == 'APPROVED WITH MINOR CONDITIONS OR MODIFICATIONS'
for u in ["Unanimous", "1 Nay", ">1 Nays"]:
    tbl += f" & {(df.loc[mask, 'unanimity']==u).sum():,.0f}"
tbl += f" & {mask.sum():,.0f}"
tbl += f" & ({mask.sum()/len(df):.1%}) \\\\ [1ex]\n"

# Approved with major conditions
tbl += " ~ ~ Approved with major conditions"
mask = df['project_result'] == 'APPROVED WITH MAJOR CONDITIONS OR MODIFICATIONS'
for u in ["Unanimous", "1 Nay", ">1 Nays"]:
    tbl += f" & {(df.loc[mask, 'unanimity']==u).sum():,.0f}"
tbl += f" & {mask.sum():,.0f}"
tbl += f" & ({mask.sum()/len(df):.1%}) \\\\ [1ex]\n"

print(header + tbl + footer)


\begin{table}[H]
\caption{Summary of Motion Outcomes and Vote Results}
\vspace{0.2cm}
\label{tab_result_unanimity}
\begin{adjustbox}{max width=\textwidth}
\begin{threeparttable}
\begin{tabular}{lrrrrr} \toprule
 & \multicolumn{3}{c}{Unanimity} & & \\
 Project Implication & Unanimous & 1 Nay & >1 Nays & Total &  \\ \midrule
  ~ ~ Approved & 281 & 15 & 3 & 299 & (41.1%) \\ [1ex]
 ~ ~ Approved with minor conditions & 247 & 20 & 12 & 279 & (38.4%) \\ [1ex]
 ~ ~ Approved with major conditions & 20 & 2 & 5 & 27 & (3.7%) \\ [1ex]

\bottomrule
\end{tabular}
\begin{tablenotes}
\item {\textit{Notes: } This table shows the number of cases decided on by the City Planning Commission, organized by the implication of the motion for the project proposal and the unanimity of the vote.}
\end{tablenotes}
\end{threeparttable}
\end{adjustbox}
\end{table}



In [ ]:
# Motion results vs. unanimity

df['unanimity'] = ''
df.loc[ df['num_noes']==0, 'unanimity'] = 'Unanimous'
df.loc[ df['num_noes']==1, 'unanimity'] = '1 Nay'
df.loc[ df['num_noes']>1, 'unanimity'] = '>1 Nays'

summ_tbl = pd.crosstab(df['project_result'], df['unanimity'])
summ_tbl.index =['Approved', 'Approved in part or with conditions', 'Deliberations continued to future date', 'Denied']

CasesDeniedPct = f"{(summ_tbl.iloc[3].sum() / summ_tbl.sum().sum()) * 100:.0f}\\%"
CasesContinuedPct = f"{(summ_tbl.iloc[2].sum() / summ_tbl.sum().sum()) * 100:.0f}\\%"
CasesApprovedWithModsPct = f"{(summ_tbl.iloc[1].sum() / summ_tbl.sum().sum()) * 100:.0f}\\%"
CasesUnanimousPct = f"{(summ_tbl.loc[:, 'Unanimous'].sum() / summ_tbl.sum().sum()) * 100:.0f}\\%"

RESULTS['CasesDeniedPct'] = CasesDeniedPct
RESULTS['CasesContinuedPct'] = CasesContinuedPct
RESULTS['CasesApprovedWithModsPct'] = CasesApprovedWithModsPct
RESULTS['CasesUnanimousPct'] = CasesUnanimousPct

HEADER = """\\begin{tabular}{lrrrrr} \\toprule
 & \\multicolumn{3}{c}{Unanimity} &  & \\\\ 
Project Implication & Unanimous & 1 Nay & >1 Nays & Total &  \\\\ \\midrule"""
FOOTER = """\\bottomrule
\\end{tabular}"""
tbl = []
for idx, row in summ_tbl.iterrows():
    unanimous = f"{row['Unanimous']:,.0f}"
    one_nay = f"{row['1 Nay']:,.0f}"
    multiple_nays = f"{row['>1 Nays']:,.0f}"
    rowsum = f"{row.sum():,.0f}"
    pct = f"({row.sum()/summ_tbl.sum().sum()*100:.1f}\\%)"
    tbl.append([f"~ ~ {idx}", unanimous, one_nay, multiple_nays, rowsum, pct])
unanimous = f"{summ_tbl['Unanimous'].sum():,.0f}"
one_nay = f"{summ_tbl['1 Nay'].sum():,.0f}"
multiple_nays = f"{summ_tbl['>1 Nays'].sum():,.0f}"
total = f"{summ_tbl.sum().sum():,.0f}"
tbl.append([f"~ TOTAL", unanimous, one_nay, multiple_nays, total, ""])
unanimous_pct = f"({summ_tbl['Unanimous'].sum()/summ_tbl.sum().sum()*100:.1f}\\%)"
one_nay_pct = f"({summ_tbl['1 Nay'].sum()/summ_tbl.sum().sum()*100:.1f}\\%)"
multiple_nays_pct = f"({summ_tbl['>1 Nays'].sum()/summ_tbl.sum().sum()*100:.1f}\\%)"
tbl.append(["", unanimous_pct, one_nay_pct, multiple_nays_pct, "", ""])
my_table = wt.latex_table(tbl, header=HEADER, footer=FOOTER)

filename = os.path.join(LOCAL_PATH, 'tables', 'tab_result_unanimity.tex')
with open(filename, 'w') as f:
    f.write(my_table)

print(my_table)

In [ ]:
summ_tbl.index =['Approved', 'Approved in part or with conditions', 'Deliberations continued to future date', 'Denied']
summ_tbl

In [ ]:
plt.figure(figsize=(6,4))
idx1 = df['n__support'] > 0
idx2 = df['n__oppose'] > 0
bins1 = np.arange(0, 10, 1)
bins2 = np.arange(-9, 1, 1)
plt.hist(np.log2(df.loc[idx1, 'n__support']), alpha=0.8, color='blue', label='supporting', bins=bins1)
plt.hist(-np.log2(df.loc[idx2, 'n__oppose']), alpha=0.8, color='orange', label='opposing', bins=bins2)
plt.xlabel(r'$\log_2$(# Letters)')
plt.ylabel('# Cases')
plt.legend()
plt.grid(axis='y', alpha=0.2)
plt.gca().set_axisbelow(True)
def custom_formatter(xval, pos):
    if xval<0:
        return f"{abs(int(xval))}"
    else:
        return f"{int(xval)}"
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(custom_formatter))
#plt.title("Distribution of Support and Opposition Letters by Project")
plt.savefig(os.path.join(LOCAL_PATH, 'figures', 'fig_support_oppose.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
wt.update_results(RESULTS)